# 라이브러리 호출

In [2]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

# duckdb
import duckdb

# 데이터 전처리
import pandas as pd
import numpy as np
import polars as pl

# 데이터 수집
import requests
from bs4 import BeautifulSoup

# ETF 구성 종목 필터링 
- 작업 이유 : 검색 키워드 설정 (구성종목)

In [3]:
ETF_conn = duckdb.connect('../DB/ETF.db')

In [4]:
ETF_df = ETF_conn.execute('select * from IRP_ETF_COMPOSE_table').fetchdf()
ETF_conn.close()

In [5]:
ETF_df = ETF_df.map(lambda x : x.strip())

In [6]:
ETF_df_task_1 = ETF_df[ETF_df['구성종목 종목명'] != "설정현금액"]
ETF_df_task_1 = ETF_df_task_1[ETF_df_task_1['구성종목 종목명'] != "원화현금"]
ETF_df_task_1 = ETF_df_task_1[1:]

In [7]:
ETF_df_task_1.head()

,기준일,ETF 코드(ISIN 코드),ETF 심볼(제로인 코드),ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율,수정자,수정일시
3,20250619,KR70000D0009,0000D0,TIGER 엔비디아미국채커버드콜밸런스(합성),KRYZTRSECJ05,KEDI Nvidia U.S. 30Y Treasury Target Cov,0.248286,FEP_INSERT,2025-06-19 08:00:08
4,20250619,KR70000D0009,0000D0,TIGER 엔비디아미국채커버드콜밸런스(합성),KRYZTRSECJ06,KEDI Nvidia U.S. 30Y Treasury Target Cov,-1.607317,FEP_INSERT,2025-06-19 08:00:08
5,20250619,KR70000D0009,0000D0,TIGER 엔비디아미국채커버드콜밸런스(합성),KRYZTRSECJ07,KEDI Nvidia U.S. 30Y Treasury Target Cov,-0.193683,FEP_INSERT,2025-06-19 08:00:08
6,20250619,KR70000D0009,0000D0,TIGER 엔비디아미국채커버드콜밸런스(합성),KRYZTRSECJ08,KEDI Nvidia U.S. 30Y Treasury Target Cov,-0.647702,FEP_INSERT,2025-06-19 08:00:08
8,20250619,KR70000H0005,0000H0,KODEX 인도Nifty미드캡100,INE002L01015,SJVN Ltd,0.248478,FEP_INSERT,2025-06-19 08:00:08


In [8]:
# 원하는 컬럼 필터링
ETF_df_task_2 = ETF_df_task_1[['ETF 종목명','구성종목 표준코드','구성종목 종목명','편입비율']]

In [9]:
# 구성종목 중 상위 5개 추출
ETF_df_task_3 = ETF_df_task_2.sort_values(by=['ETF 종목명','편입비율'], ascending=False)

In [10]:
# 종목별 상위 5개
ETF_df_task_4= ETF_df_task_3.groupby('ETF 종목명').head(5)

In [11]:
ETF_df_task_4

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
7839,파워 코스피100,KR7005930003,삼성전자,22.979477
7899,파워 코스피100,KR7105560007,KB금융,2.846638
7856,파워 코스피100,KR7012450003,한화에어로스페이스,2.680652
7876,파워 코스피100,KR7035420009,NAVER,2.595262
7836,파워 코스피100,KR7005380001,현대차,2.280681
...,...,...,...,...
59727,1Q 25-08 회사채(A+이상)액티브,KR6079317D93,JB 우리캐피탈488-3(지),8.851804
59728,1Q 25-08 회사채(A+이상)액티브,KR6095923D95,현대커머셜485-3(지),8.850334
59723,1Q 25-08 회사채(A+이상)액티브,KR6023788D91,신한캐피탈487-2,8.844456
59722,1Q 25-08 회사채(A+이상)액티브,KR601945CD90,아이비케이캐피탈290-7,8.842961


In [12]:
ETF_df_task_4['구성종목 종목명'].value_counts()

구성종목 종목명
삼성전자                            86
한화에어로스페이스                       85
KB금융                            84
NAVER                           78
SK하이닉스                          73
                                ..
MITSUBISHI HEAVY INDUSTRIES      1
TAKEDA PHARMACEUTICAL CO LTD     1
MITSUBISHI CORP                  1
부국증권 20250604-26-4(단)            1
현대카드886-3                        1
Name: count, Length: 1741, dtype: int64

In [13]:
ETF_compose_ticker = ETF_df_task_4['구성종목 종목명'].value_counts().reset_index()['구성종목 종목명'].unique()

In [14]:
len(ETF_compose_ticker)

1741

### 키워드 검색 후 뉴스기사 가져오기

In [15]:
# userAgent포함
header = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"}

In [16]:
def extract_href(data):
    return data.attrs['href']

In [17]:
def extract_headline(url):
    temp_headline = BeautifulSoup(requests.get(url,headers = header).text,'html.parser')
    return temp_headline.select_one('h1.headline').text.strip()

In [18]:
def extract_content(url):
    temp_content = BeautifulSoup(requests.get(url,headers = header).text,'html.parser')
    return temp_content.select_one('#articletxt').text.strip()

In [19]:
def extract_summary(url):
    temp_summary = BeautifulSoup(requests.get(url,headers = header).text,'html.parser')

    if(temp_summary.select_one('#container > div > div > article > div > div > div.article-body-wrap > div.summary')):
        result_summary = temp_summary.select_one('#container > div > div > article > div > div > div.article-body-wrap > div.summary').text.strip()
        return result_summary
    else:
        return '요약 없음'

In [20]:
def extract_real_url(data):
    temp_list = data.split('/')
    if temp_list[-2] == 'article':
        return data

In [21]:
def extract_datetime(url):
    temp_datetime = BeautifulSoup(requests.get(url,headers = header).text,'html.parser')
    return temp_datetime.select_one('#container div.content div.datetime span.txt-date').text.strip()

In [187]:
def get_list(url):
    response = requests.get(url,headers = header)
    a_tag_list = BeautifulSoup(response.text).select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul').find_all('a')
    url_lst_before = [a_tag.attrs['href'] for a_tag in a_tag_list]
    url_lst_after = set(list(filter(None,list(map(extract_real_url,url_lst_before)))))
    return url_lst_after

In [ ]:
 def extract_news_data(query_text,page_range):
    request_base_url = 'https://search.hankyung.com/search/news?query={query}&page={pages}'
    query_text = query_text
    wanted_pages = page_range
    
    ingest_url_lst = [request_base_url.format(query=query_text,pages=page+1) for page in range(wanted_pages)]
    
    # 현재 페이지 호출용도
    page_lst = [i+1 for i in range(page_range)]
    
    # 페이지당 뉴스 기사 목록들 저장 리스트
    news_list = []

    
    for number, url in enumerate(ingest_url_lst):
        try:
            url_list= get_list(url)
            #print(url_list)
            # 수집 보여주기용 데이터 리스트
            #show_list = []
            
            for target in url_list:
                # 기사 내용 저장 용도 변수
                news_dict = dict()
                news_dict['header'] = extract_headline(target)
                news_dict['summary'] = extract_summary(target)
                news_dict['content'] = extract_content(target)
                news_dict['url'] = target
                news_dict['datetime'] = extract_datetime(target)
                
                news_list.append(news_dict)
                #show_list.append(news_dict)
                #print(news_dict)
            #show_df = pd.DataFrame(show_list)
            print(f'########### 수집한 기사 목록 - page : {page_lst[number]}   ############')    
        # display(show_df)
        except AttributeError :
           other_dict = {'header' : None, 'summary' : None,'content' : None, 'url':None,'datetime':None}
           news_list.append(other_dict)
           
    save_df = pd.DataFrame(news_list)
    save_df.to_csv(f'../data/{query_text}_target_news.csv',encoding='utf-8-sig')

In [189]:
extract_news_data('NAVER',1)

########### 수집한 기사 목록 - page : 1   ############


In [192]:
pd.read_csv('../data/NAVER_target_news.csv',index_col=[0])

,header,summary,content,url,datetime
0,이자 부담 최소화! 증권사 신용대출 3%대로 바꾸는 절호의 기회,요약 없음,"전송종목 : 삼화콘덴서, 아세아, 경농, 아세아제지, 넥센타이어최근 주식 투자자들 ...",https://www.hankyung.com/article/202507248245a,2025.07.24 14:23
1,"24일, 기관 거래소에서 삼성전자(-0.6%), KODEX 레버리지(-0.28%) ...",요약 없음,"기관 투자자는 24일 거래소에서 삼성전자, KODEX 레버리지, 삼성에스디에스 등을...",https://www.hankyung.com/article/202507240574L,2025.07.24 18:35
2,최휘영 문체부장관 후보자 장녀 '아빠 찬스' 의혹,父 대표 지낸 네이버 美자회사 취업\n2019년 2월 영주권 취득…12월 퇴사\n청...,사진=뉴스1\n\n 최휘영 문화체육관광부 장관 후보자의 ...,https://www.hankyung.com/article/2025072400687,2025.07.24 16:18
3,"동문건설, 25일 '춘천 동문 디 이스트 어반포레' 견본주택 개관","지상 29층, 6개 동, 569가구 규모\n교통·편의·생활·자연 갖춘 입지 강점",'춘천 동문 디 이스트 어반포레' 투시도. 동문건설 제공\n\n ...,https://www.hankyung.com/article/202507249840i,2025.07.24 15:07
4,"""데이터센터에 미술품 의무…황당 규제 풀어야""","기업들, 과기부와 간담회서 \n주차장·전력수급 등 애로 호소","정부가 ‘인공지능(AI) 고속도로’ 사업을 적극적으로 추진하고 있지만 전력 수급, ...",https://www.hankyung.com/article/2025072403371,2025.07.24 17:31
5,'케이스퀘어데이터센터 가산' 준공,"현대건설, 지하 3층~지상 11층 서울 남부권 디지털 허브 구축",현대건설이 지속적인 데이터센터 건설로 디지털 생태계 확장을 위한 핵심 기지 구축에 ...,https://www.hankyung.com/article/2025072403671,2025.07.24 17:32
6,"24일, 외국인 거래소에서 NAVER(-1.94%), 에이피알(-3.01%) 등 순매도",요약 없음,"외국인 투자자는 24일 거래소에서 NAVER, 에이피알, 현대건설 등을 중점적으로 ...",https://www.hankyung.com/article/202507240573L,2025.07.24 18:35
7,"이정은 ""조정석 엄마 하기엔 너무 젊죠?"" [인터뷰+]",필감성 감독 '좀비딸'로 돌아온 이정은 \n'만화 찢고 나온' 할머니 밤순 역 연기...,"/사진=NEW\n\n ""조정석(44) 엄마 역할 하기엔 ...",https://www.hankyung.com/article/202507249176H,2025.07.24 14:07
8,"현대건설, 서울 남부권 디지털 허브 '케이스퀘어데이터센터 가산' 준공","최적의 시공 전략, 차별화된 솔루션\n디지털 시대 핵심 인프라 구축\n지하 3층~지...",'케이스퀘어데이터센터 가산' 조감도./현대건설 제공 \n\n현대건설이 지속적인 데이...,https://www.hankyung.com/article/202507249690i,2025.07.24 14:44
9,"카카오 ""LLM 韓 1위""…SKT ""수학·코딩 향상""","한국형 LLM 성능 경쟁 치열\n\n기업들, 오픈소스 모델 잇단 공개\n정부 'K인...",국내 테크 기업들 사이에서 대규모언어모델(LLM) 성능 경쟁이 치열해지고 있다. 정...,https://www.hankyung.com/article/2025072403421,2025.07.24 17:33


keyword마다 뉴스 형태가 달라서, 오류가 날 수 있음. 오류가 날 경우, 추가 대응 필요

# 리팩토링 이후

In [30]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ▶ 헤더
headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"}

# ▶ 기사 리스트 파싱 함수
async def get_article_urls(session, page_url):
    try:
        async with session.get(page_url, headers=headers) as resp:
            text = await resp.text()
            soup = BeautifulSoup(text, 'html.parser')
            ul = soup.select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul')
            if not ul:
                return []

            urls = []
            for a in ul.find_all('a', href=True):
                href = a['href']
                if '/article/' in href:
                    urls.append(href)
            return list(set(urls))  # 중복 제거
    except Exception as e:
        print(f"[get_article_urls error] {page_url} - {e}")
        return []

# ▶ 기사 상세 파싱 함수
async def fetch_article(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            html = await resp.text()
            soup = BeautifulSoup(html, 'html.parser')

            return {
                'header': soup.select_one('h1.headline').text.strip() if soup.select_one('h1.headline') else None,
                'summary': soup.select_one('div.summary').text.strip() if soup.select_one('div.summary') else None,
                'content': soup.select_one('#articletxt').text.strip() if soup.select_one('#articletxt') else None,
                'url': url,
                'datetime': soup.select_one('div.datetime span.txt-date').text.strip() if soup.select_one('div.datetime span.txt-date') else None,
            }
    except Exception as e:
        print(f"[fetch_article error] {url} - {e}")
        return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}

# ▶ 메인 비동기 루프
async def extract_news_data_async(query_text, page_range):
    base_url = 'https://search.hankyung.com/search/news?query={query}&page={page}'
    search_urls = [base_url.format(query=query_text, page=p+1) for p in range(page_range)]

    async with aiohttp.ClientSession() as session:
        # 1. 페이지별 기사 링크 수집
        tasks = [get_article_urls(session, url) for url in search_urls]
        results = await asyncio.gather(*tasks)
        article_urls = list(set([url for sublist in results for url in sublist]))

        print(f"🔗 총 {len(article_urls)}개의 기사 URL 수집됨")

        # 2. 기사 본문 수집
        article_tasks = [fetch_article(session, url) for url in article_urls]
        articles = await asyncio.gather(*article_tasks)

        # 3. 저장
        df = pd.DataFrame(articles)
        save_word = query_text.replace('/','_')
        df.to_csv(f'../data/{save_word}_target_news.csv', encoding='utf-8-sig', index=False)
        print(f"✅ 저장 완료: ../data/{save_word}_target_news.csv")

# ▶ 실행 함수
def extract_news_data(query_text, page_range):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(extract_news_data_async(query_text, page_range))

In [ ]:
# 50페이지씩 수집
for ticker in tqdm(ETF_compose_ticker):
    if f'{ticker}_target_news.csv' in os.listdir('../data'):
        continue
    extract_news_data(ticker,50)

  0%|          | 0/1741 [00:00<?, ?it/s]

🔗 총 12개의 기사 URL 수집됨


  1%|          | 20/1741 [00:02<04:12,  6.82it/s]

✅ 저장 완료: ../data/NVIDIA Corp_target_news.csv
🔗 총 57개의 기사 URL 수집됨


  1%|▏         | 24/1741 [00:08<11:22,  2.51it/s]

...

✅ 저장 완료: ../data/BNK금융지주_target_news.csv


  5%|▌         | 93/1741 [04:42<5:28:11, 11.95s/it]

🔗 총 0개의 기사 URL 수집됨
✅ 저장 완료: ../data/TENCENT HOLDINGS LTD_target_news.csv
🔗 총 500개의 기사 URL 수집됨


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x103f7d940>>
Traceback (most recent call last):
  File "/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
  5%|▌         | 94/1741 [04:58<6:01:56, 13.19s/it]

✅ 저장 완료: ../data/SKC_target_news.csv
🔗 총 500개의 기사 URL 수집됨


# duckdb에 저장하기

In [15]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
import duckdb

# ▶ 헤더
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

# ▶ 기사 리스트 파싱 함수
async def get_article_urls(session, page_url):
    try:
        async with session.get(page_url, headers=headers) as resp:
            text = await resp.text()
            soup = BeautifulSoup(text, 'html.parser')
            ul = soup.select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul')
            if not ul:
                return []

            urls = []
            for a in ul.find_all('a', href=True):
                href = a['href']
                if '/article/' in href:
                    urls.append(href)
            return list(set(urls))  # 중복 제거
    except Exception as e:
        print(f"[get_article_urls error] {page_url} - {e}")
        return []

# ▶ 기사 상세 파싱 함수
async def fetch_article(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            html = await resp.text()
            soup = BeautifulSoup(html, 'html.parser')

            hostname = urlparse(url).hostname

            # ✅ 1. magazine.hankyung.com용 로직
            if 'magazine.hankyung.com' in hostname:
                return {
                    'header': soup.select_one('#contents h1.news-tit').text.strip() if soup.select_one('#contents h1.news-tit') else None,
                    'summary': None,
                    'content': soup.select_one('#magazineView').text.strip() if soup.select_one('#magazineView') else None,
                    'url': url,
                    'datetime': soup.select_one('#contents span.txt-num').text.strip() if soup.select_one('#contents span.txt-num') else None,
                }

            # ✅ 2. www.hankyung.com일 경우 기존 로직
            elif 'hankyung.com' in hostname:
                return {
                    'header': soup.select_one('h1.headline').text.strip() if soup.select_one('h1.headline') else None,
                    'summary': soup.select_one('div.summary').text.strip() if soup.select_one('div.summary') else None,
                    'content': soup.select_one('#articletxt').text.strip() if soup.select_one('#articletxt') else None,
                    'url': url,
                    'datetime': soup.select_one('div.datetime span.txt-date').text.strip() if soup.select_one('div.datetime span.txt-date') else None,
                }

            # ✅ 알 수 없는 도메인
            else:
                print(f"⚠️ 알 수 없는 호스트: {hostname}")
                return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}

    except Exception as e:
        print(f"[fetch_article error] {url} - {e}")
        return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}
# ▶ 메인 비동기 루프
async def extract_news_data_async(query_text, page_range):
    base_url = 'https://search.hankyung.com/search/news?query={query}&page={page}'
    search_urls = [base_url.format(query=query_text, page=p+1) for p in range(page_range)]

    async with aiohttp.ClientSession() as session:
        # 1. 페이지별 기사 링크 수집
        tasks = [get_article_urls(session, url) for url in search_urls]
        results = await asyncio.gather(*tasks)
        article_urls = list(set([url for sublist in results for url in sublist]))

        print(f"🔗 총 {len(article_urls)}개의 기사 URL 수집됨")

        # 2. 기사 본문 수집
        article_tasks = [fetch_article(session, url) for url in article_urls]
        articles = await asyncio.gather(*article_tasks)

        # 3. ticker 컬럼 추가
        for article in articles:
            article['ticker'] = query_text

        # 4. 비어 있으면 dummy row 추가
        if not articles:
            articles = [{
                'header': None,
                'summary': None,
                'content': None,
                'url': None,
                'datetime': None,
                'ticker': query_text
            }]
            print("⚠️ 수집된 기사가 없어 None 값으로 대체 저장합니다.")

        # 4. DuckDB 저장
        df = pd.DataFrame(articles)
        con = duckdb.connect('../DB/news_articles.db')

        # Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
        con.register('temp_df', df)

        # 테이블이 없다면 생성
        con.execute("""
            CREATE TABLE IF NOT EXISTS articles AS
            SELECT * FROM temp_df LIMIT 0
        """)

        # 데이터 삽입
        con.execute("INSERT INTO articles SELECT * FROM temp_df")

        # 정리
        con.unregister('temp_df')
        con.close()

        print(f"✅ 저장 완료: ../DB/news_articles.db (ticker = {query_text})")

# ▶ 실행 함수
def extract_news_data(query_text, page_range):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(extract_news_data_async(query_text, page_range))

In [ ]:
con = duckdb.connect('../DB/news_articles.db')
con.execute("DROP TABLE IF EXISTS articles")
con.close()

NameError: name 'duckdb' is not defined

In [ ]:
# 50페이지씩 수집
for ticker in tqdm(ETF_compose_ticker):
    extract_news_data(ticker,50)

  0%|          | 0/1741 [00:00<?, ?it/s]

🔗 총 500개의 기사 URL 수집됨


  0%|          | 1/1741 [00:14<6:53:36, 14.26s/it]

✅ 저장 완료: ../DB/news_articles.db (ticker = 삼성전자)
🔗 총 500개의 기사 URL 수집됨


  0%|          | 2/1741 [00:28<6:51:25, 14.20s/it]

...

✅ 저장 완료: ../DB/news_articles.db (ticker = ADVANCED MICRO DEVICES)
🔗 총 500개의 기사 URL 수집됨


  2%|▏         | 36/1741 [07:41<6:04:10, 12.82s/it]


KeyboardInterrupt: 

[fetch_article error] https://magazine.hankyung.com/business/article/202505317693b - Connector is closed.
[fetch_article error] https://magazine.hankyung.com/business/article/202505294924b - Connector is closed.
[fetch_article error] https://magazine.hankyung.com/business/article/202505282111b - Connector is closed.
[fetch_article error] https://magazine.hankyung.com/business/article/202503256176b - Connector is closed.


: 

In [15]:
con  = duckdb.connect('../DB/news_articles.db')

In [16]:
news_df = con.execute('select * from articles').fetch_df()

In [18]:
news_df

,header,summary,content,url,datetime,ticker
0,관세 협상 앞두고 이재용·김동관·최태원 만난 李…무슨 얘기 나눴나,None,이재명 대통령이 6월 13일 서울 용산 대통령실청사에서 열린 6경제단체와 기업인 간...,https://magazine.hankyung.com/business/article...,2025.07.25 08:04,삼성전자
1,"SKT, 위약금 16만명 이탈…이젠 고객 쟁탈전",유심해킹 사고 이후 총 80만명 이탈 \n면제 이후 이탈자 예상보다 적어 \n폴더블...,영상 모듈 닫기\n\n\n\n\n<앵커>SK텔레콤이 해킹 사고 이후 위약금을 면제해...,https://www.hankyung.com/article/2025071501305,2025.07.15 17:23,삼성전자
2,"미국, 중국산 흑연에 반덤핑 관세 예고…2차전지 관련주 급등",None,미국이 중국산 흑연에 고율의 반덤핑 관세를 예고하면서 국내 2차전지 업종 주가가 일...,https://www.hankyung.com/article/202507211741a,2025.07.22 11:00,삼성전자
3,"갤럭시 워치8 시리즈, 가장 얇은 디자인…'웨어러블 혁신'의 새 기준 제시",삼성전자 '갤럭시 언팩 2025'\n\n뛰어난 착용감과 강력한 기능\n차별화된 쿠션...,‘갤럭시 워치8 클래식 46㎜ 화이트’. 삼성전자 제공\n\n\n ...,https://www.hankyung.com/article/2025072235641,2025.07.22 16:39,삼성전자
4,9~10% 주식 관련 대출에서 3%대 대출로 교체,None,"전송종목 : 하이트진로, 유한양행, CJ대한통운, 두산, DL최근 주식 투자자들 사...",https://www.hankyung.com/article/202507185994a,2025.07.18 08:33,삼성전자
...,...,...,...,...,...,...
11050,"미 마이크론·웨스턴디지털, 키옥시아 지분 거래 타진",None,미국 반도체 기업인 마이크론(Micron)과 웨스턴 디지털(Western Digit...,https://www.hankyung.com/article/202104018917Y,2021.04.01 15:48,ADVANCED MICRO DEVICES
11051,"NH투자증권, ELS 6종 모집…최대 연 12% 수익 추구",None,NH투자증권은 주가연계파생결합증권(ELS) 6종을 내달 1일 오후 1시까지 모집한다...,https://www.hankyung.com/article/2021033080886,2021.03.30 12:54,ADVANCED MICRO DEVICES
11052,美증시 주요지수 일제히 하락…다우 0.48%↓,None,사진=게티이미지뱅크 \n\n 뉴욕 증시가 일부 기업들 실...,https://www.hankyung.com/article/2017102633887,2017.10.26 06:28,ADVANCED MICRO DEVICES
11053,"한국투자증권, 테슬라·AMD-엔비디아 기초 ELS 2종 공모",None,한국투자증권은 뱅키스 전용 주가연계증권(ELS) 2종을 각각 50억원 한도로 공모한...,https://www.hankyung.com/article/2022060294825,2022.06.02 13:59,ADVANCED MICRO DEVICES
